<a href="https://colab.research.google.com/github/LonelyGuy-SE1/QLoRA-Test/blob/main/QLoRA_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [57]:
!pip install unsloth transformers datasets accelerate bitsandbytes

In [58]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-1.5B",
    max_seq_length = 2048,
    load_in_4bit = True,   # THIS is where quantization happens
)

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [59]:
from datasets import load_dataset

dataset = load_dataset("RaffaSch121/fixed_spider")
dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)

train_ds = dataset["train"]
test_ds  = dataset["test"]

In [60]:
def build_schema(example):
    return example["db_schema"]

In [61]:
def generate_sql(question, schema):
    prompt = f"""You are an expert SQL generator.

Given the database schema, write a correct SQL query.

### Rules:
- Use ONLY the tables and columns provided
- Do NOT invent table names
- Do NOT use prefixes like dev.
- Return ONLY the SQL query

### Schema:
{schema}

### Question:
{question}

### SQL:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text.split("### SQL:")[-1].strip()

In [62]:
def exact_match(pred, gold):
    return pred.strip().lower() == gold.strip().lower()

In [63]:
results = []

samples = test_ds.select(range(50))   # keep small for now

for ex in samples:
    schema = build_schema(ex)
    pred = generate_sql(ex["question"], schema)
    score = exact_match(pred, ex["query"])
    results.append(score)

baseline_acc = sum(results) / len(results)
print("Baseline:", baseline_acc)

Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1

Baseline: 0.02


In [64]:
ex = samples[0]
pred = generate_sql(ex["question"], ex["db_schema"])

print("Question:", ex["question"])
print("Prediction:", pred)
print("Ground Truth:", ex["query"])

Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: how many inhabitants does boulder have
Prediction: SELECT COUNT(*) FROM dev.city WHERE city_name = 'boulder'
Ground Truth: SELECT population FROM city WHERE city_name  =  "boulder";


Qlora Prep

In [65]:
def format_example(example):
    return {
        "text": f"""You are an expert SQL generator.

### Schema:
{example["db_schema"]}

### Question:
{example["question"]}

### SQL:
{example["query"]}
"""
    }

train_formatted = train_ds.map(format_example)

In [66]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.00,
)

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 0 O layers and 0 MLP layers.


In [73]:
def tokenize(example):
    out = tokenizer(
        example["text"],
        truncation=True,
        padding="longest",   # 🔥 instead of max_length
        max_length=512,
    )
    out["labels"] = out["input_ids"].copy()
    return out

In [74]:
from transformers import TrainingArguments, Trainer

trainer = Trainer(
    model=model,
    train_dataset=train_tokenized,
    args=TrainingArguments(
        per_device_train_batch_size=6,
        gradient_accumulation_steps=2,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        output_dir="outputs",
        dataloader_num_workers=2,
        remove_unused_columns=False,   # 🔥 important
    ),
)

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,753 | Num Epochs = 2 | Total steps = 1,294
O^O/ \_/ \    Batch size per device = 6 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (6 x 2 x 1) = 12
 "-____-"     Trainable parameters = 1,089,536 of 1,544,803,840 (0.07% trained)


Step,Training Loss
10,4.743922
20,5.498696
30,5.703191
40,5.429716
50,5.008294
60,5.042818
70,5.610612
80,5.851115
90,4.428799
100,4.864613


In [ ]:
results = []

samples = test_ds.select(range(50))   # keep small for now

for ex in samples:
    schema = build_schema(ex)
    pred = generate_sql(ex["question"], schema)
    score = exact_match(pred, ex["query"])
    results.append(score)

baseline_acc = sum(results) / len(results)
print("Baseline:", baseline_acc)